# Chia tập phân tầng, chuẩn hóa, kiểm tra rò rỉ, chọn 9 đặc trưng

**Đầu vào:** `output/wine_clean.csv`

**Đầu ra (bàn giao):** `output/split_data.npz`. Ngoài ra có `X_train.csv`, `X_test.csv`, `y_train.csv`, `y_test.csv`, bản chuẩn hóa và bản 9 đặc trưng. Chạy từ thư mục `notebooks/`.

In [1]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

os.makedirs("../output", exist_ok=True)
pd.set_option("display.max_colwidth", None)
pd.set_option("display.width", 200)

from scipy.stats import spearmanr
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.inspection import permutation_importance

RANDOM_STATE = 42
label_names = {0: "Thấp (≤5)", 1: "Trung bình (6)", 2: "Cao (≥7)"}

In [ ]:
# ------------------------------------------------------------------ đọc đầu vào dữ liệu
CLEAN = "../output/wine_clean.csv"
if os.path.exists(CLEAN):
    wine_clean = pd.read_csv(CLEAN)
    print("Đọc", CLEAN)
else:
    # tạo lại đúng như việc 3: gộp, loại trùng, mã hóa wine_type, giữ ngoại lai
    red = pd.read_csv("../data/winequality-red.csv", sep=";")
    white = pd.read_csv("../data/winequality-white.csv", sep=";")
    red["wine_type"], white["wine_type"] = "red", "white"
    wine_clean = pd.concat([red, white], ignore_index=True).drop_duplicates().reset_index(drop=True)
    wine_clean["wine_type"] = wine_clean["wine_type"].map({"red": 0, "white": 1})
    wine_clean.to_csv(CLEAN, index=False)
    print("Chưa có wine_clean.csv, đã tạo lại từ data/")

assert wine_clean.duplicated().sum() == 0 and wine_clean.isnull().sum().sum() == 0
print("Shape:", wine_clean.shape)
print("Cột:", wine_clean.columns.tolist())

Chưa có wine_clean.csv, đã tạo lại từ data/
Shape: (5320, 13)
Cột: ['fixed acidity', 'volatile acidity', 'citric acid', 'residual sugar', 'chlorides', 'free sulfur dioxide', 'total sulfur dioxide', 'density', 'pH', 'sulphates', 'alcohol', 'quality', 'wine_type']


## 1. Tạo nhãn 3 lớp và chia tập huấn luyện / kiểm tra phân tầng

Nhãn: thấp (quality ≤ 5) = 0, trung bình (= 6) = 1, cao (≥ 7) = 2. Chia 80% huấn luyện / 20% kiểm tra, **phân tầng theo tổ hợp (nhãn, loại rượu)** vì nhãn mất cân bằng và tỷ lệ nhãn khác nhau giữa rượu đỏ và trắng; `random_state = 42`.

In [3]:
def quality_to_class(q):
    return 0 if q <= 5 else (1 if q == 6 else 2)


wine_clean["quality_class"] = wine_clean["quality"].apply(quality_to_class)
wine_clean["stratify_col"] = wine_clean["quality_class"].astype(str) + "_" + wine_clean["wine_type"].astype(str)

train_df, test_df = train_test_split(wine_clean, test_size=0.2, stratify=wine_clean["stratify_col"],
                                     random_state=RANDOM_STATE)
print("Train:", train_df.shape, "| Test:", test_df.shape)


def pct(s):
    return (s.value_counts(normalize=True).sort_index() * 100).round(2)


check = pd.DataFrame({
    "Toàn bộ": pct(wine_clean["stratify_col"]),
    "Huấn luyện": pct(train_df["stratify_col"]),
    "Kiểm tra": pct(test_df["stratify_col"]),
})
check.index = [f"{label_names[int(k[0])]} – {'đỏ' if k[-1] == '0' else 'trắng'}" for k in check.index]
print("\n=== Tỷ lệ (%) từng tổ hợp (nhãn, loại rượu) ===")
print(check.to_string())

Train: (4256, 15) | Test: (1064, 15)

=== Tỷ lệ (%) từng tổ hợp (nhãn, loại rượu) ===
                        Toàn bộ  Huấn luyện  Kiểm tra
Thấp (≤5) – đỏ            12.03       12.03     12.03
Thấp (≤5) – trắng         25.34       25.33     25.38
Trung bình (6) – đỏ       10.06       10.06     10.06
Trung bình (6) – trắng    33.61       33.62     33.55
Cao (≥7) – đỏ              3.46        3.45      3.48
Cao (≥7) – trắng          15.51       15.51     15.51


In [4]:
# Tách đặc trưng / nhãn; quality và cột phân tầng KHÔNG được làm đầu vào
drop_cols = ["quality", "quality_class", "stratify_col"]
X_train, y_train = train_df.drop(columns=drop_cols), train_df["quality_class"]
X_test, y_test = test_df.drop(columns=drop_cols), test_df["quality_class"]
assert "quality" not in X_train.columns and "quality_class" not in X_train.columns

for name, obj in [("X_train", X_train), ("X_test", X_test), ("y_train", y_train), ("y_test", y_test)]:
    obj.to_csv(f"../output/{name}.csv", index=False)
print("Đã lưu: output/X_train.csv, X_test.csv, y_train.csv, y_test.csv")
print("Đặc trưng:", X_train.columns.tolist())

Đã lưu: output/X_train.csv, X_test.csv, y_train.csv, y_test.csv
Đặc trưng: ['fixed acidity', 'volatile acidity', 'citric acid', 'residual sugar', 'chlorides', 'free sulfur dioxide', 'total sulfur dioxide', 'density', 'pH', 'sulphates', 'alcohol', 'wine_type']


## 2. Mã hóa, chuẩn hóa chỉ khớp trên tập huấn luyện

- **Mã hóa:** chỉ có `wine_type` là biến phân loại, nhị phân nên mã 0/1 (đã làm ở việc 3), không cần one-hot.
- **Chuẩn hóa:** mô hình cây (Decision Tree, Bagging, Random Forest, Extra Trees) chia theo ngưỡng nên không cần. Vẫn tạo bản z-score cho các mô hình cơ sở tuyến tính hoặc dựa khoảng cách; `StandardScaler` **chỉ khớp (fit) trên tập huấn luyện**, rồi dùng để biến đổi tập kiểm tra. `wine_type` không chuẩn hóa vì là nhị phân.

In [5]:
num_cols = [c for c in X_train.columns if c != "wine_type"]
scaler = StandardScaler().fit(X_train[num_cols])
X_train_s = X_train.copy()
X_train_s[num_cols] = scaler.transform(X_train[num_cols])
X_test_s = X_test.copy()
X_test_s[num_cols] = scaler.transform(X_test[num_cols])
X_train_s.to_csv("../output/X_train_scaled.csv", index=False)
X_test_s.to_csv("../output/X_test_scaled.csv", index=False)

# train có trung bình ≈ 0, độ lệch chuẩn ≈ 1; test lệch nhẹ là bình thường vì scaler không khớp trên test
print("Trung bình train sau chuẩn hóa (tối đa |.|):", round(float(X_train_s[num_cols].mean().abs().max()), 6))
print("Độ lệch chuẩn train sau chuẩn hóa:", sorted(set(X_train_s[num_cols].std(ddof=0).round(3))))
print("Trung bình test sau chuẩn hóa (tối đa |.|):", round(float(X_test_s[num_cols].mean().abs().max()), 3))

Trung bình train sau chuẩn hóa (tối đa |.|): 0.0
Độ lệch chuẩn train sau chuẩn hóa: [1.0]
Trung bình test sau chuẩn hóa (tối đa |.|): 0.04


## 3. Kiểm tra rò rỉ dữ liệu

In [6]:
overlap = pd.merge(X_train.drop_duplicates(), X_test.drop_duplicates(), how="inner").shape[0]
leak = pd.DataFrame([
    ("Bản ghi trùng giữa train và test", overlap, "Đã loại trùng trước khi chia (việc 3)"),
    ("Cột quality / quality_class trong đặc trưng", int(any(c in X_train.columns for c in ("quality", "quality_class"))),
     "Nhãn chỉ nằm trong y"),
    ("Scaler khớp trên test", 0, f"scaler.n_samples_seen_ = {scaler.n_samples_seen_} = số mẫu train ({len(X_train)})"),
], columns=["Kiểm tra", "Số vi phạm", "Ghi chú"])
assert overlap == 0 and scaler.n_samples_seen_ == len(X_train)
print(leak.to_string(index=False))


def dist(y):
    return " / ".join(f"{v:.1f}%" for v in y.value_counts(normalize=True).sort_index() * 100)


sizes = pd.DataFrame({
    "Tập dữ liệu": ["Huấn luyện", "Kiểm tra", "Tổng"],
    "Số mẫu": [len(X_train), len(X_test), len(wine_clean)],
    "Tỷ lệ": [f"{len(X_train) / len(wine_clean) * 100:.0f}%", f"{len(X_test) / len(wine_clean) * 100:.0f}%", "100%"],
    "Số đặc trưng": [X_train.shape[1]] * 3,
    "Phân bố lớp thấp / TB / cao": [dist(y_train), dist(y_test), dist(wine_clean["quality_class"])],
})
print("\n=== Kích thước và phân bố lớp các tập ===")
print(sizes.to_string(index=False))

                                   Kiểm tra  Số vi phạm                                             Ghi chú
           Bản ghi trùng giữa train và test           0               Đã loại trùng trước khi chia (việc 3)
Cột quality / quality_class trong đặc trưng           0                                Nhãn chỉ nằm trong y
                      Scaler khớp trên test           0 scaler.n_samples_seen_ = 4256 = số mẫu train (4256)



=== Kích thước và phân bố lớp các tập ===
Tập dữ liệu  Số mẫu Tỷ lệ  Số đặc trưng Phân bố lớp thấp / TB / cao
 Huấn luyện    4256   80%            12       37.4% / 43.7% / 19.0%
   Kiểm tra    1064   20%            12       37.4% / 43.6% / 19.0%
       Tổng    5320  100%            12       37.4% / 43.7% / 19.0%


## 4. Chọn đặc trưng (chỉ dùng tập huấn luyện)

Ba bước: (1) tương quan Spearman với nhãn và giữa các đặc trưng; (2) permutation importance và F1 macro cross-validation khi bỏ từng đặc trưng; (3) kiểm chứng loại đồng thời các đặc trưng yếu.

In [7]:
feats = X_train.columns.tolist()

# (1) Tương quan Spearman với nhãn và các cặp đặc trưng tương quan cao
corr_label = pd.Series({f: spearmanr(X_train[f], y_train)[0] for f in feats}).sort_values(key=abs, ascending=False)
print("=== (1) Tương quan Spearman với nhãn ===")
print(corr_label.round(3).to_string())

cm = X_train.corr(method="spearman")
pairs = [(a, b, round(cm.loc[a, b], 3)) for i, a in enumerate(feats) for b in feats[i + 1:] if abs(cm.loc[a, b]) >= 0.7]
print("\nCác cặp đặc trưng tương quan cao (|r| >= 0,7):")
print(pd.DataFrame(pairs, columns=["Đặc trưng A", "Đặc trưng B", "Spearman r"]).to_string(index=False))

=== (1) Tương quan Spearman với nhãn ===
alcohol                 0.495
density                -0.355
chlorides              -0.313
volatile acidity       -0.255
wine_type               0.124
citric acid             0.104
fixed acidity          -0.100
free sulfur dioxide     0.077
total sulfur dioxide   -0.072
pH                      0.056
residual sugar         -0.034
sulphates               0.025

Các cặp đặc trưng tương quan cao (|r| >= 0,7):
        Đặc trưng A          Đặc trưng B  Spearman r
free sulfur dioxide total sulfur dioxide       0.741


In [8]:
# (2a) Permutation importance trên phần validation tách từ tập train
X_tr, X_val, y_tr, y_val = train_test_split(X_train, y_train, test_size=0.25, stratify=y_train,
                                            random_state=RANDOM_STATE)
rf = RandomForestClassifier(n_estimators=300, random_state=RANDOM_STATE, n_jobs=-1).fit(X_tr, y_tr)
perm = permutation_importance(rf, X_val, y_val, scoring="f1_macro", n_repeats=10,
                              random_state=RANDOM_STATE, n_jobs=-1)
perm_imp = pd.Series(perm.importances_mean, index=feats)

# (2b) F1 macro cross-validation khi bỏ lần lượt từng đặc trưng
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)


def cv_f1(cols):
    m = RandomForestClassifier(n_estimators=200, random_state=RANDOM_STATE, n_jobs=-1)
    return cross_val_score(m, X_train[cols], y_train, cv=cv, scoring="f1_macro").mean()


base = cv_f1(feats)
delta = pd.Series({f: cv_f1([c for c in feats if c != f]) - base for f in feats})
print(f"F1 macro CV với đủ {len(feats)} đặc trưng: {base:.4f}")

summary_fs = pd.DataFrame({"Tương quan với nhãn": corr_label, "Permutation importance": perm_imp,
                           "Δ F1 khi bỏ": delta}).round(4)
# giữ nếu permutation importance >= 0,01 hoặc bỏ đi làm F1 giảm
keep = (summary_fs["Permutation importance"] >= 0.01) | (summary_fs["Δ F1 khi bỏ"] < -0.001)
summary_fs["Đề xuất"] = np.where(keep, "giữ", "xem xét loại")
summary_fs = summary_fs.sort_values("Permutation importance", ascending=False)
print("\n=== (2) Bảng tổng hợp chọn đặc trưng ===")
print(summary_fs.to_string())
summary_fs.to_csv("../output/bang_chon_dac_trung.csv")

F1 macro CV với đủ 12 đặc trưng: 0.5819

=== (2) Bảng tổng hợp chọn đặc trưng ===
                      Tương quan với nhãn  Permutation importance  Δ F1 khi bỏ       Đề xuất
alcohol                            0.4954                  0.1274      -0.0002           giữ
volatile acidity                  -0.2551                  0.0458      -0.0036           giữ
chlorides                         -0.3129                  0.0317      -0.0025           giữ
free sulfur dioxide                0.0772                  0.0289      -0.0063           giữ
density                           -0.3549                  0.0253       0.0079           giữ
total sulfur dioxide              -0.0724                  0.0214      -0.0068           giữ
citric acid                        0.1036                  0.0171      -0.0005           giữ
pH                                 0.0564                  0.0162      -0.0056           giữ
sulphates                          0.0253                  0.0157      -0.0054   

In [9]:
# (3) Kiểm chứng loại ĐỒNG THỜI các đặc trưng yếu; lặp 3 cách chia fold (15 fold) để giảm nhiễu
def cv_f1_multi(cols, seeds=(42, 7, 2024)):
    scores = []
    for sd in seeds:
        cv_ = StratifiedKFold(n_splits=5, shuffle=True, random_state=sd)
        m = RandomForestClassifier(n_estimators=200, random_state=RANDOM_STATE, n_jobs=-1)
        scores.append(cross_val_score(m, X_train[cols], y_train, cv=cv_, scoring="f1_macro"))
    s = np.concatenate(scores)
    return s.mean(), s.std()


weak = ["fixed acidity", "residual sugar", "wine_type"]
selected_features = [f for f in feats if f not in weak]
compare_fs = pd.DataFrame([("Đủ 12 đặc trưng", *cv_f1_multi(feats)),
                           (f"Bỏ {len(weak)} đặc trưng yếu (còn {len(selected_features)})", *cv_f1_multi(selected_features))],
                          columns=["Tập đặc trưng", "F1 macro CV (trung bình)", "Độ lệch chuẩn (15 fold)"]).round(4)
print("=== (3) Kiểm chứng loại đồng thời ===")
print(compare_fs.to_string(index=False))
print("\nĐặc trưng được giữ:", selected_features)

=== (3) Kiểm chứng loại đồng thời ===
             Tập đặc trưng  F1 macro CV (trung bình)  Độ lệch chuẩn (15 fold)
           Đủ 12 đặc trưng                    0.5815                   0.0184
Bỏ 3 đặc trưng yếu (còn 9)                    0.5821                   0.0154

Đặc trưng được giữ: ['volatile acidity', 'citric acid', 'chlorides', 'free sulfur dioxide', 'total sulfur dioxide', 'density', 'pH', 'sulphates', 'alcohol']


**Kết luận chọn đặc trưng:** `fixed acidity`, `residual sugar` và `wine_type` có permutation importance dưới 0,01 và bỏ từng cái không làm giảm F1. Loại cả ba cùng lúc, F1 macro CV gần như không đổi (khoảng 0,582 ở cả hai cách) và độ lệch chuẩn giữa các fold nhỏ hơn. Vì vậy giữ **9 đặc trưng**: volatile acidity, citric acid, chlorides, free sulfur dioxide, total sulfur dioxide, density, pH, sulphates, alcohol. `density` vẫn được giữ dù tương quan mạnh với `alcohol` (|r| ≈ 0,69), vì permutation importance của nó không nhỏ. `wine_type` không làm đầu vào mô hình nhưng vẫn được lưu kèm để phân tích theo loại rượu.

## 5. Bàn giao `split_data.npz`

In [10]:
X_train[selected_features].to_csv("../output/X_train_selected.csv", index=False)
X_test[selected_features].to_csv("../output/X_test_selected.csv", index=False)

# 9 đặc trưng đã chọn (chưa chuẩn hóa, mô hình cây không cần), nhãn 0/1/2 và loại rượu
np.savez("../output/split_data.npz",
         X_train=X_train[selected_features].to_numpy(), X_test=X_test[selected_features].to_numpy(),
         y_train=y_train.to_numpy(), y_test=y_test.to_numpy(),
         wine_type_train=X_train["wine_type"].to_numpy(), wine_type_test=X_test["wine_type"].to_numpy(),
         feature_names=np.array(selected_features))

# đọc lại để kiểm tra file bàn giao
d = np.load("../output/split_data.npz")
for k in d.files:
    print(f"{k:16s} {d[k].shape}")
print("Đã lưu: output/split_data.npz")

X_train          (4256, 9)
X_test           (1064, 9)
y_train          (4256,)
y_test           (1064,)
wine_type_train  (4256,)
wine_type_test   (1064,)
feature_names    (9,)
Đã lưu: output/split_data.npz


**Cách dùng ở bước huấn luyện (việc 5, 6):**

```python
import numpy as np
d = np.load("../output/split_data.npz")
X_train, y_train = d["X_train"], d["y_train"]
X_test, y_test = d["X_test"], d["y_test"]   # chỉ dùng một lần ở bước đánh giá cuối
features = list(d["feature_names"])
```